In [ ]:
import threading
import time

import numpy as np
import pyarrow as pa
import ursid as us

s = us.local()
book = s.input_table(
    {"id": us.int64(), "sym": us.string(), "price": us.float64(), "size": us.int64()},
    key="id",
)
syms = np.array(["AAPL", "MSFT", "NVDA", "TSLA", "GOOG", "AMZN", "META", "JPM"])
rng = np.random.default_rng(1)


def feed():
    while True:
        n = 2_000
        book.upsert(
            pa.table(
                {
                    "id": rng.integers(0, 100_000, n),
                    "sym": syms[rng.integers(0, len(syms), n)],
                    "price": rng.normal(100, 10, n).round(2),
                    "size": rng.integers(1, 1000, n),
                }
            )
        )
        time.sleep(0.1)


threading.Thread(target=feed, daemon=True).start()
time.sleep(0.5)

In [ ]:
book

In [ ]:
n = 5_000_000
big = s.static_table(
    pa.table(
        {
            "i": np.arange(n),
            "x": rng.normal(size=n),
            "sym": syms[rng.integers(0, len(syms), n)],
        }
    )
)
big

In [ ]:
book.sort("price desc")